# <a id='toc1_'></a>[Tempo de chegada e desvio de feixe](#toc0_)

**Sumário**<a id='toc0_'></a>    
- [Tempo de chegada e desvio de feixe](#toc1_)    
  - [Introdução](#toc1_1_)    
  - [Desenvolvimento](#toc1_2_)    
    - [Tempo de chegada](#toc1_2_1_)    
  - [Conclusão](#toc1_3_)    
  - [Referências](#toc1_4_)    

<!-- vscode-jupyter-toc-config
	numbering=false
	anchor=true
	flat=false
	minLevel=1
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

## <a id='toc1_1_'></a>[Introdução](#toc0_)

Por fim, vamos aplicar todas as funções que montamos para analisar qual foi a interferência real causada por um terremoto e armazenar os dados obtidos em arquivos para podermos utilizar posteriormente.

## <a id='toc1_2_'></a>[Desenvolvimento](#toc0_)

### <a id='toc1_2_1_'></a>[Tempo de chegada](#toc0_)

Vamos considerar um dos terremotos do Panamá

In [ ]:
from obspy.clients.fdsn import Client
import obspy.core.utcdatetime as utc
from Scripts.module import analise_catalog

catalog = "USP"
client = Client(catalog)
events = client.get_events(
    starttime=utc.UTCDateTime("2026-10-09T00:00:00.000000Z"),
    endtime=utc.UTCDateTime("2026-10-09T23:59:00.000000Z"),
    minmagnitude=7.0,
)

event_data, time_data = analise_catalog(events, catalog, plotting=False)

In [214]:
event_data

,event_id,latitude,longitude,epicenter_distance,depth,magnitude,magnitude_type,Source,origin_time,receptor_latitude,receptor_longitude
0,Panama2026-10-09T17:56:07.118061Z,7.194002,-81.318901,44.986691,12.6,7.66938,mB,USP,2026-10-09T17:56:07.118061Z,-22.807606,-47.052526


In [215]:
time_data

,event_id,phase,takeoff_angle,frequency,incidente_angle,ray_parameter,travel_time,arrival_time,model_used
0,Panama2026-10-09T17:56:07.118061Z,P,24.580301,Unknown,24.528479,456.015855,494.886675,2026-10-09T18:04:22.004736Z,iasp91
1,Panama2026-10-09T17:56:07.118061Z,pP,155.397993,Unknown,24.550134,456.393508,498.838260,2026-10-09T18:04:25.956321Z,iasp91
2,Panama2026-10-09T17:56:07.118061Z,sP,166.046287,Unknown,24.546198,456.324870,500.502155,2026-10-09T18:04:27.620216Z,iasp91
3,Panama2026-10-09T17:56:07.118061Z,PcP,10.395651,Unknown,10.374863,197.816936,595.751170,2026-10-09T18:06:02.869231Z,iasp91
4,Panama2026-10-09T17:56:07.118061Z,PP,33.754895,Unknown,33.6792,609.136112,600.121780,2026-10-09T18:06:07.239841Z,iasp91
...,...,...,...,...,...,...,...,...,...
28,Panama2026-10-09T17:56:07.118061Z,PKIKKIKS,2.840006,Unknown,1.641543,54.317362,2101.279903,2026-10-09T18:31:08.397964Z,iasp91
29,Panama2026-10-09T17:56:07.118061Z,SKIKKIKS,1.569158,Unknown,1.566054,51.820115,2312.880564,2026-10-09T18:34:39.998625Z,iasp91
30,Panama2026-10-09T17:56:07.118061Z,PKIKPPKIKP,3.777634,Unknown,3.770152,72.227477,2392.649096,2026-10-09T18:35:59.767157Z,iasp91
31,Panama2026-10-09T17:56:07.118061Z,PKPPKP,13.058158,Unknown,13.031877,247.692530,2456.711519,2026-10-09T18:37:03.829580Z,iasp91


Agora, vamos buscar os dados do intervalo de tempo próximo aos previstos pelo modelo. Vamos adicionar um acréscimo de uma hora antes e depois para garantir que conseguimos toda a informação necessária.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from datetime import datetime, timezone, timedelta
from Scripts.module import get_archiver_data, PV_NAMES

pv = PV_NAMES["BPM_X"]

first_arrival = min(time_data["arrival_time"])
last_arrival = max(time_data["arrival_time"])
from_time = first_arrival - timedelta(minutes=30)
to_time = last_arrival + timedelta(minutes=30)
time_delta = int(to_time - from_time)

df = get_archiver_data(
    pv=pv,
    from_time=from_time,
    to_time=to_time,
    optimize=time_delta,
)

df.to_csv("Data/panama_example.csv")

# Utilize o código abaixo para carregar os dados caso não tenha acesso à API
# df = pd.read_csv("Data/panama_example.csv", index_col=0)

df = df.sort_values("secs")
df = df[(df["secs"] >= from_time) & (df["secs"] <= to_time)]
df

,secs,val,nanos,severity,status,std,min,max,count
2,1791567263,76.582364,500000000,0,0,92.459427,7414.831373,7722.871811,11
3,1791567264,32.772600,500000000,0,0,98.014526,7347.118944,7676.143850,10
4,1791567265,90.186229,500000000,0,0,89.627949,7463.863996,7748.212791,10
5,1791567266,153.890598,500000000,0,0,95.569506,7501.302522,7806.916652,11
6,1791567267,132.734459,500000000,0,0,93.203578,7532.411981,7842.941997,10
...,...,...,...,...,...,...,...,...,...
6347,1791573608,-61.085812,500000000,0,0,157.940079,7256.690889,7728.095707,11
6348,1791573609,147.701221,500000000,0,0,181.637267,7348.904083,7969.361666,10
6349,1791573610,554.162595,500000000,0,0,170.402594,7841.604850,8352.620501,11
6350,1791573611,857.814374,500000000,0,0,192.834157,8063.981010,8650.900284,10


Vamos escrever uma função para plotar o gráfico com as ondas

In [254]:
import numpy as np
import plotly.graph_objects as go
from obspy import UTCDateTime

def plot_signal(
    signal: np.asarray,
    time_data: pd.DataFrame,
    regions: np.asarray = None,
    plot_time_data: bool = True
):
    fig = go.Figure()
    fig.update_layout(
        autosize=False,
        width=900, 
        height=500,
        legend=dict(
            groupclick="togglegroup"
        )
    )

    fig.add_scatter(x=signal["secs"].to_list(), y=signal["val"].to_list(), name="Sinal")
    
    u_max = np.max(signal["val"].to_list())
    u_min = np.min(signal["val"].to_list())
    
    for i, region in enumerate(regions if regions != None else []):
        fig.add_trace(
            go.Scatter(
                x=[region["x0"], region["x0"], region["x1"], region["x1"], region["x0"]],
                y=[region["y0"], region["y1"], region["y1"], region["y0"], region["y0"]],
                fill="toself",
                opacity=0.6,
                name=region["name"],
                legendgroup=region["name"],
                line_width=2,
            )
        )
    
    if not plot_time_data:
        fig.show()
        return

    for name in np.unique(time_data["event_id"].to_list()):
        time_list = [UTCDateTime(t).timestamp for t in time_data[time_data["event_id"] == name]["arrival_time"].to_list()]
        colors_list = [
            "red" if phase == "love" else ("blue" if phase == "rayleigh" else "green")
            for phase in
            time_data[time_data["event_id"] == name]["phase"].to_list()
        ]

        for i, j in enumerate(time_list):
            fig.add_trace(
                go.Scatter(
                    x=[j, j],
                    y=[u_min, u_max],
                    mode="lines",
                    line=dict(color=colors_list[i], width=2, dash="dash"),
                    name=f"MAG {event_data[event_data["event_id"] == name]["magnitude"].values[0]:.2f} - {name}",
                    legendgroup=name,
                    showlegend=(i == 0),
                )
            )
            
    fig.show()
    
plot_signal(df, time_data)

Agora, precisamos definir uma forma de calcular a amplitude de um determinado sinal. Para isso, vamos começar estimando o nível do ruído das partes iniciais, andando com uma janela de tempo e considerando o desvio padrão médio dos sinais dessas janelas.

In [255]:

def get_initial_noise(signal, time_window=60, window_step=10, step_count=50, start_time=0):
    noise_std = []
    min_noise = float("inf")
    max_noise = -float("inf")
    
    regions = []
    
    from_time = df.iloc[start_time]["secs"]
    to_time = df.iloc[start_time]["secs"] + time_window

    for _ in range(step_count):
        df_window = df[(df["secs"] >= from_time) & (df["secs"] <= to_time)]["val"]
        # plot_signal(df_window, time_data, False) 
        
        min_noise = min(min_noise, np.min(df_window))   
        max_noise = max(max_noise, np.max(df_window))   
        
        noise_std.append(np.std(df_window))
        regions.append((from_time, to_time))
    
        from_time = from_time + window_step
        to_time = to_time + window_step
    
    mean_noise_std = np.mean(noise_std)
        
    return min_noise, max_noise, mean_noise_std, regions
        
min_noise, max_noise, noise_mean_std, noise_regions = get_initial_noise(df)

noise_region = {
    "x0": noise_regions[0][0],
    "x1": noise_regions[-1][1],
    "y0": min_noise-noise_mean_std,
    "y1": max_noise+noise_mean_std,
    "name": "Noise region"
}

plot_signal(df, time_data, regions=[noise_region])

Agora, vamos calcular (de forma grosseira) a amplitude total do espaço entre os tempos de chegada. Aqui, vamos considerar que toda a amplitude gerada dentro da janela de chegadas é resultado do terremoto sendo analisado, o que quase nunca é verdade. Pode ser que outros terremotos ou fenômenos também interfiram nesse tempo, mas vamos ignorar isso.

Para considerar os erros dos tempos de chegada, vamos considerar uma janela com 1 minuto antes e depois da janela de chegadas.

Como métricas, vamos calcular a amplitude total e o desvio padrão do sinal

In [262]:
def get_signal_amplitude(signal, first_arrival, last_arrival):
    window_start = first_arrival - timedelta(minutes=1)
    window_end = last_arrival + timedelta(minutes=1)
    
    signal_window = signal[(signal["secs"] >= window_start) & (signal["secs"] <= window_end)]
    
    min_signal = np.min(signal_window["val"])
    max_signal = np.max(signal_window["val"])
    
    std = np.std(signal_window["val"])
    
    region = {
        "x0": window_start.timestamp,
        "x1": window_end.timestamp,
        "y0": min_signal,
        "y1": max_signal,
        "name": "Amplitude region"
    }
    
    return min_signal, max_signal, std, region

min_signal, max_signal, signal_std, signal_region = get_signal_amplitude(df, first_arrival, last_arrival)

plot_signal(df, time_data, regions=[signal_region])

Assim, conseguimos estimar uma caixa que contém a variação sentida. Vamos comparar essa variação com a padrão do ruído

In [268]:
factor_std = signal_std / noise_mean_std
factor_amplitude = (max_signal - min_signal) / (max_noise - min_noise)

print(f"Fator de diferença sinal/ruído do desvio padrão: {factor_std:.2f}x")
print(f"Fator de diferença sinal/ruído da amplitude: {factor_amplitude:.2f}x")

plot_signal(df, time_data, regions=[noise_region, signal_region])

Fator de diferença sinal/ruído do desvio padrão: 84.56x
Fator de diferença sinal/ruído da amplitude: 146.90x


## <a id='toc1_3_'></a>[Conclusão](#toc0_)

Assim, conseguimos extrair vagamente o impacto do terremoto no feixe. É claro que existirão casos em que esse dado não vai estar completamente correto e corente, mas vamos começar assim para ver se os modelos utilizados conseguem abstrair pelo menos parte da informação.

<span id="referencias"></span>
## <a id='toc1_4_'></a>[Referências](#toc0_)

1. PLOTLY. *Plotly Python Graphing Library*. [S. l.]: Plotly, [s. d.]. Disponível em: https://plotly.com/python/.